In [0]:
%run "/Workspace/Users/rahulpatel@cyntexa.com/DataEngineering-Project/de_project/src/includes"

In [0]:
# dbutils.widgets.text("catalog", "de_dev")

In [0]:
catalog = dbutils.widgets.get("catalog")
print(catalog)

In [0]:
df = spark.table(f"{catalog}.bronze.sales")
clean_df = (
    df
    .dropna(subset=["sale_id" , "customer_id","product_id","quantity","sale_amount","sale_date","payment_method","order_status"])
    .fillna({"discount":0})
    .dropDuplicates(["sale_id"])
    .select(
        "sale_id" , "customer_id","product_id","quantity","sale_amount","discount","sale_date",
        "region","payment_method","order_status"
    )
    )
clean_df.createOrReplaceTempView("sales_clean_view")
# display(clean_df)


In [0]:
%sql
CREATE TABLE IF NOT EXISTS ${catalog}.silver.sales_scd_1 (
    sale_id INT,
    customer_id int,
    product_id int,
    quantity int,
    sale_amount decimal(12,2),
    discount decimal(5,2),
    sale_date date,
    region string,
    payment_method string,
    order_status string
)
USING DELTA;

In [0]:
%sql
MERGE INTO ${catalog}.silver.sales_scd_1 AS trg
USING sales_clean_view AS src
ON trg.sale_id = src.sale_id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *  

In [0]:
%sql
-- select * from ${catalog}.silver.sales_scd_1 limit 100;